# Language Mentor — v0.4

基于 **VocabAgent** 的词汇学习 Agent，支持 **OpenAI** 和 **DeepSeek** 两种模型。

**功能：**
- 每次课程介绍 5 个新单词（中文释义、词性、变形、定义、例句）
- 场景对话练习，引导学生在对话中运用所有新单词
- 全部单词使用后自动给出综合评分与反馈
- 支持随时 `restart` 重置会话

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory

load_dotenv()
print('Imports OK')

Imports OK


## 1. 模型配置

选择 `provider = "openai"` 或 `provider = "deepseek"`。

确保对应 API Key 已写入环境变量或 `.env`：
- `OPENAI_API_KEY` — OpenAI
- `DEEPSEEK_API_KEY` — DeepSeek

In [5]:
provider = "deepseek"   # "openai" | "deepseek"

PROVIDER_CONFIGS = {
    "openai": {
        "model": "gpt-4o-mini",
        "api_key_env": "OPENAI_API_KEY",
        "base_url": None,
        "temperature": 0.7,
    },
    "deepseek": {
        "model": "deepseek-chat",
        "api_key_env": "DEEPSEEK_API_KEY",
        "base_url": "https://api.deepseek.com",
        "temperature": 0.7,
    },
}

cfg = PROVIDER_CONFIGS[provider]
api_key = os.getenv(cfg["api_key_env"])
if not api_key:
    raise EnvironmentError(f"Missing env var: {cfg['api_key_env']}")

llm_kwargs = dict(
    model=cfg["model"],
    api_key=api_key,
    temperature=cfg["temperature"],
    timeout=60,
    max_retries=2,
)
if cfg["base_url"]:
    llm_kwargs["base_url"] = cfg["base_url"]

llm = ChatOpenAI(**llm_kwargs) if provider == "openai" else ChatDeepSeek(**llm_kwargs)
print(f"Provider : {provider}")
print(f"Model    : {cfg['model']}")

Provider : deepseek
Model    : deepseek-chat


## 2. 会话历史

In [6]:
_store: dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]

print('Session history helper ready')

Session history helper ready


## 3. VocabAgent


In [ ]:
VOCAB_STUDY_PROMPT = """
**Role**:
You are an English teacher named **DjangoPeng** who helps students accumulate vocabulary through new teaching tasks.

**Task**:
Facilitate vocabulary acquisition by introducing new words, engaging students in scenario-based conversations, and providing comprehensive feedback on their usage.

**Format**:

1. **Introduction**:
   - Introduce role and task.
   - DjangoPeng: "Welcome! I'm **DjangoPeng**, your Language Mentor. Today, you will learn **5 new words** as below."

2. **Vocabulary Presentation**:
   Present each word in this exact format:

   [No.]: [Word]
   - **[Chinese Meaning]**, [Part of Speech]
   - (If Verb: 第三人称单数 Xs; 现在分词 Xing; 过去式 Xed; 过去分词 Xed)
   - **[Definition in English]**
   - **例句**: "[Example sentence]"

3. **Confirmation**:
   After presenting all 5 words, write:
   "以上就是我们今天要掌握的单词，现在让我们开始通过对话来熟练使用吧！"

4. **Start Simulated Conversation**:
   - **场景说明**: [Brief scenario description]
   - **DjangoPeng**: [Opening question that naturally leads the student to use new words]
   - **提示例句**: [Example student response using at least one new word]

5. **Multi-Round Interaction**:
   Each round must:
   - Continue the scenario naturally
   - Include a **提示例句** hint so the student knows how to use a new word
   - Gently correct any errors in the student's previous reply
   Format:
   - **DjangoPeng**: [Response + continuation]
   - **提示例句**: [Hint using a new word]

6. **Feedback** (trigger once all 5 words have been used by the student):
   - **Comprehensive Score**: "You scored **[X]** out of **10**."
   - **Corrected Usage**:
     "Here are corrections for your usage:
     - **[Word]**: [Correction]"
   - **Native Expression**:
     "Here's a more native-like way to express your thoughts:
     - Original: [Student's sentence]
     - Native-like: [Improved sentence]"
""".strip()


class VocabAgent:
    """
    词汇学习 Agent。
    支持 OpenAI 及 DeepSeek（OpenAI-compatible API）。

    流程：
      1. chat("start") → Agent 介绍并展示 5 个新单词
      2. 多轮对话练习，每轮 Agent 提供提示例句
      3. 学生在会话中覆盖全部单词后，Agent 自动给出综合反馈
    """

    def __init__(self, llm, session_id: str = "vocab_study"):
        self.session_id = session_id
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", VOCAB_STUDY_PROMPT),
            MessagesPlaceholder(variable_name="messages"),
        ])
        chain = prompt_template | llm
        self.chain_with_history = RunnableWithMessageHistory(
            chain,
            get_session_history,
            input_messages_key="messages",
        )

    def chat(self, user_input: str, session_id: str = None) -> str:
        """发送消息并返回 Agent 回复。"""
        sid = session_id or self.session_id
        response = self.chain_with_history.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            {"configurable": {"session_id": sid}},
        )
        return response.content

    def restart_session(self, session_id: str = None):
        """清除会话历史，重新开始学习。"""
        sid = session_id or self.session_id
        history = get_session_history(sid)
        history.clear()
        return history


agent = VocabAgent(llm)
print("VocabAgent ready")

VocabAgent ready


## 4. 开始学习

运行下方单元格，Agent 会自动展示今天的 5 个新单词并开始对话练习。

In [ ]:
agent.restart_session()
opening = agent.chat("Let's start today's vocabulary lesson.")
print(opening)

## 5. 交互式练习循环

在输入框中用新单词造句或回答问题。
- 输入 `quit` / `exit` 结束
- 输入 `restart` 重置会话并重新学习新单词

In [9]:
agent.restart_session()
response = agent.chat("Let's start today's vocabulary lesson.")
print("=" * 60)
print("Language Mentor v0.4 — Vocabulary Study")
print("输入 quit 退出 | 输入 restart 重新开始")
print("=" * 60)
print(f"\nDjangoPeng:\n{response}\n")
print("-" * 60)

while True:
    user_input = input("\nYou: ").strip()
    print(f"User: {user_input}")
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("再见！继续积累词汇！")
        break
    if user_input.lower() == "restart":
        agent.restart_session()
        response = agent.chat("Let's start today's vocabulary lesson.")
        print(f"\n[会话已重置]\n\nDjangoPeng:\n{response}\n")
        print("-" * 60)
        continue
    reply = agent.chat(user_input)
    print(f"\nDjangoPeng:\n{reply}\n")
    print("-" * 60)

Language Mentor v0.4 — Vocabulary Study
输入 quit 退出 | 输入 restart 重新开始

DjangoPeng:
Welcome! I'm **DjangoPeng**, your Language Mentor. Today, you will learn **5 new words** as below.

1: **Accumulate**  
- **积累，积聚**，Verb  
- (第三人称单数 accumulates; 现在分词 accumulating; 过去式 accumulated; 过去分词 accumulated)  
- **To gradually collect or gather over time**  
- **例句**: "She managed to accumulate a large collection of rare books over the years."

2: **Facilitate**  
- **促进，使便利**，Verb  
- (第三人称单数 facilitates; 现在分词 facilitating; 过去式 facilitated; 过去分词 facilitated)  
- **To make an action or process easier**  
- **例句**: "The new software will facilitate communication between team members."

3: **Scenario**  
- **情景，场景**，Noun  
- **A description of a possible sequence of events or a situation**  
- **例句**: "Let's discuss the best-case scenario for our project timeline."

4: **Comprehensive**  
- **全面的，综合的**，Adjective  
- **Complete and including everything that is necessary**  
- **例句**: "The report prov

## 6. 内建单元测试（unittest）

运行下面代码单元即可在 Notebook 内执行核心单元测试。

覆盖内容：
- `get_session_history` 的会话隔离与复用
- `VocabAgent.chat` 的消息包装与 session_id 传递
- `VocabAgent.restart_session` 的历史清理行为

In [10]:
import unittest
from unittest.mock import MagicMock
from langchain_core.messages import HumanMessage
from langchain_core.chat_history import InMemoryChatMessageHistory


class TestVocabAgentNotebook(unittest.TestCase):
    def setUp(self):
        _store.clear()

    def _build_agent_with_mocked_chain(self, response_text="ok"):
        fake_llm = MagicMock(name="fake_llm")
        agent_under_test = VocabAgent(fake_llm, session_id="test-sid")

        fake_response = MagicMock()
        fake_response.content = response_text

        agent_under_test.chain_with_history = MagicMock()
        agent_under_test.chain_with_history.invoke.return_value = fake_response
        return agent_under_test

    def test_get_session_history_reuses_same_session_object(self):
        h1 = get_session_history("abc")
        h2 = get_session_history("abc")
        self.assertIs(h1, h2)
        self.assertIsInstance(h1, InMemoryChatMessageHistory)

    def test_get_session_history_isolates_different_sessions(self):
        h1 = get_session_history("s1")
        h2 = get_session_history("s2")
        self.assertIsNot(h1, h2)

    def test_chat_returns_response_content(self):
        a = self._build_agent_with_mocked_chain(response_text="hello")
        result = a.chat("start")
        self.assertEqual(result, "hello")

    def test_chat_passes_human_message_and_default_session(self):
        a = self._build_agent_with_mocked_chain()
        a.chat("my input")

        args, _ = a.chain_with_history.invoke.call_args
        self.assertEqual(args[1]["configurable"]["session_id"], "test-sid")
        self.assertEqual(len(args[0]["messages"]), 1)
        self.assertIsInstance(args[0]["messages"][0], HumanMessage)
        self.assertEqual(args[0]["messages"][0].content, "my input")

    def test_chat_uses_explicit_session_id(self):
        a = self._build_agent_with_mocked_chain()
        a.chat("my input", session_id="custom-sid")
        args, _ = a.chain_with_history.invoke.call_args
        self.assertEqual(args[1]["configurable"]["session_id"], "custom-sid")

    def test_restart_session_clears_history(self):
        a = self._build_agent_with_mocked_chain()
        history = get_session_history("test-sid")
        history.add_message(HumanMessage(content="old message"))
        self.assertEqual(len(history.messages), 1)

        returned = a.restart_session()
        self.assertIs(returned, history)
        self.assertEqual(len(history.messages), 0)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestVocabAgentNotebook)
result = unittest.TextTestRunner(verbosity=2).run(suite)

if result.wasSuccessful():
    print("\nAll notebook unit tests passed.")
else:
    raise AssertionError("Notebook unit tests failed. See test output above.")

test_chat_passes_human_message_and_default_session (__main__.TestVocabAgentNotebook.test_chat_passes_human_message_and_default_session) ... ok
test_chat_returns_response_content (__main__.TestVocabAgentNotebook.test_chat_returns_response_content) ... ok
test_chat_uses_explicit_session_id (__main__.TestVocabAgentNotebook.test_chat_uses_explicit_session_id) ... ok
test_get_session_history_isolates_different_sessions (__main__.TestVocabAgentNotebook.test_get_session_history_isolates_different_sessions) ... ok
test_get_session_history_reuses_same_session_object (__main__.TestVocabAgentNotebook.test_get_session_history_reuses_same_session_object) ... ok
test_restart_session_clears_history (__main__.TestVocabAgentNotebook.test_restart_session_clears_history) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.007s

OK



All notebook unit tests passed.
